# Stage-A Gate Weighting — the lesson that was never carried across

**Current best end-to-end: `two_stage_stageB_dynamic_v5.ipynb`, macro-F1 0.6363 / accuracy 0.9089.**
Every notebook since `two_stage_hierarchical.ipynb` has used the *same* gate, unchanged:

```python
g.fit(Xa, yb, sample_weight=compute_sample_weight("balanced", yb))
```

## The oversight

`compute_sample_weight("balanced")` assigns weights proportional to $1/n_c$ — which is **exactly
$\alpha = 1.0$** in the tempered-weighting family $w_c \propto (N/n_c)^\alpha$ used throughout Stage B.

And the Stage-B weighting study found:

> **`class_weight='balanced'` is a bad default here — it ranked 12th of 14, below no weighting at
> all.** Use $\alpha \approx 0.25$ instead: +0.0297 macro-F1 over `balanced`.

That finding was applied to Stage B and to every pool member. **It was never applied to the gate.**
Stage A has been sitting at the operating point the project explicitly rejected for Stage B.

## Why it should matter

The gate currently runs at **attack recall 0.9961 / precision 0.9195** (OOF). Recall is essentially
saturated; precision is not. About 8% of gate-positive rows are Benign, and those flows are then
*forced* to receive an attack family — they land overwhelmingly in Fuzzers, which is the model's
largest error sink (34% of its false positives are Benign leaking through the gate).

`balanced` weighting pushes a binary classifier toward the minority class, i.e. toward **recall at the
expense of precision**. Since gate recall is already 0.9961, that trade is being made in the direction
where there is nothing left to gain and real precision to lose.

## Two knobs, and why both are swept

Weighting and thresholding both move along a precision/recall curve, so they are partially redundant —
`two_stage_thresholds.ipynb` already found the gate threshold optimal at $p_{att} > 0.5$ **under
`balanced` weighting**. But weighting changes what the model *learns* (the shape of the boundary),
not merely where it is cut. Sweeping $\alpha \times t$ jointly separates the two:

- if the best $(\alpha, t)$ lies at $t = 0.5$ with $\alpha < 1$, weighting bought something
  thresholding could not;
- if every $\alpha$ reaches the same peak at some compensating $t$, the two are redundant and the
  earlier threshold finding already covered it. **That is a clean negative result, not a wasted run.**

## Protocol

Stage B is **frozen** to the v5 winner (`[DEC] DYN OLA-cw k=100 p=8`), so the gate is the only moving
part. Family predictions are computed for *every* row once; each gate configuration is then just a
different routing mask over the same predictions.

| Parameter | Fitted on |
|---|---|
| pool OOF probabilities | cached 5-fold cross-fit over `train+val` |
| gate OOF probabilities, per $\alpha$ | same 5 folds, refit per $\alpha$ |
| $W$, competence reference set | COMB attack rows |
| $(\alpha, t)$ choice | **SELV** |
| final gate | refit on all of `train+val` at the selected $\alpha$ |
| evaluation | **test**, once |

> **Runtime ~12 min** (30 gate fits dominate; cached to `cache_gate_alpha.npz`).

## 1. Setup

In [1]:
import numpy as np, pandas as pd, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
NAMES = [LABELS[i] for i in range(10)]
RS = 42

X = pd.read_csv("Resource/CIC-UNSW-Data.csv")
y = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([X, y], axis=1).drop_duplicates().reset_index(drop=True)
X, y = df[X.columns], df["Label"]
nun = X.nunique(); zero = nun[nun <= 1].index.tolist()
corr = X.drop(columns=zero).corr()
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red = [c for c in upper.columns if (upper[c] >= 0.95).any()]
X = X.drop(columns=list(dict.fromkeys(zero + red)))

X_trf, X_test, y_trf, y_test = train_test_split(X, y, test_size=0.2, random_state=RS, stratify=y)
X_trf = X_trf.reset_index(drop=True); y_trf = y_trf.reset_index(drop=True)

attack_labels = sorted(l for l in y_trf.unique() if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC = len(attack_labels)
y_arr = y_trf.values
att_trf = y_arr != 0
fam_trf = np.full(len(y_arr), -1)
fam_trf[att_trf] = pd.Series(y_arr[att_trf]).map(fam_to_idx).values

SPECS = ([ (f"XGB a={a}", ("alpha", a)) for a in (0.0, 0.25, 0.5, 1.0) ]
         + [("XGB smote2000", ("smote", 2000)),
            ("RF a=0.25", ("rf", 0.25)), ("ET a=0.25", ("et", 0.25))])
MODELS = [s[0] for s in SPECS]; M = len(MODELS)

z = np.load("cache_oof_v4.npz")
OOF = [z[f"m{m}"] for m in range(M)]
print("loaded pool OOF | train+val %d (%d attack, %.1f%%) | test %d"
      % (len(y_arr), att_trf.sum(), 100 * att_trf.mean(), len(y_test)))

loaded pool OOF | train+val 244938 (64936 attack, 26.5%) | test 61235


In [2]:
def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw)
    return XGBClassifier(**p)

def wc_alpha(n, a):
    w = (n.sum() / n) ** a
    return w / w.mean()

def sample_w(y_arr_, wc):
    w = wc[y_arr_].astype(float)
    return w * len(w) / w.sum()

def fit_member(kind, param, Xa, ya):
    n = np.bincount(np.asarray(ya), minlength=NC)
    if kind == "smote":
        strat = {c: param for c in range(NC) if n[c] < param}
        Xs, ys = SMOTE(random_state=RS, sampling_strategy=strat).fit_resample(Xa, ya)
        m = make_xgb(); m.fit(Xs, ys); return m
    if   kind == "alpha": m = make_xgb()
    elif kind == "rf":    m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    elif kind == "et":    m = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    m.fit(Xa, ya, sample_weight=sample_w(np.asarray(ya), wc_alpha(n, param)))
    return m

def fit_gate_alpha(Xa, ya10, alpha):
    yb = (ya10 != 0).astype(int)
    n  = np.bincount(yb, minlength=2)
    g  = make_xgb()
    g.fit(Xa, yb, sample_weight=sample_w(yb, wc_alpha(n, alpha)))
    return g

nb_ = np.bincount((y_arr != 0).astype(int))
print("gate class balance: benign %d / attack %d  (ratio %.2f:1)" % (nb_[0], nb_[1], nb_[0] / nb_[1]))
print("attack weight by alpha:")
for a in (0.0, 0.1, 0.25, 0.5, 0.75, 1.0):
    w = wc_alpha(nb_, a)
    print("  alpha=%.2f -> benign %.3f attack %.3f (ratio %.3f)" % (a, w[0], w[1], w[1] / w[0]))
print("NOTE: alpha=1.0 is exactly what compute_sample_weight('balanced') produces.")

gate class balance: benign 180002 / attack 64936  (ratio 2.77:1)
attack weight by alpha:
  alpha=0.00 -> benign 1.000 attack 1.000 (ratio 1.000)
  alpha=0.10 -> benign 0.949 attack 1.051 (ratio 1.107)
  alpha=0.25 -> benign 0.873 attack 1.127 (ratio 1.290)
  alpha=0.50 -> benign 0.750 attack 1.250 (ratio 1.665)
  alpha=0.75 -> benign 0.635 attack 1.365 (ratio 2.148)
  alpha=1.00 -> benign 0.530 attack 1.470 (ratio 2.772)
NOTE: alpha=1.0 is exactly what compute_sample_weight('balanced') produces.


## 2. Freeze Stage B to the v5 winner and predict every row once

Family predictions are computed for **all** rows, not just gate-positive ones. Each gate
configuration is then a different mask over the same predictions, so the sweep costs nothing extra
and the gate is provably the only thing changing.

In [3]:
comb_idx, selv_idx = train_test_split(np.arange(len(y_arr)), test_size=0.5,
                                      random_state=RS + 1, stratify=y_arr)
ref_idx = comb_idx[att_trf[comb_idx]]
y_ref = fam_trf[ref_idx]
CORRECT = np.stack([(OOF[m][ref_idx].argmax(1) == y_ref) for m in range(M)])
WC_REF = wc_alpha(np.bincount(y_ref, minlength=NC), 0.25)

prof = lambda ps: np.hstack(ps).astype(np.float32)
NN_D = NearestNeighbors(n_neighbors=100, algorithm="brute", n_jobs=-1).fit(
    prof([OOF[m][ref_idx] for m in range(M)]))

def comp_ola_cw(idx, k=100):
    nb = idx[:, :k]
    wn = WC_REF[y_ref[nb]]
    denom = wn.sum(1)
    W = np.empty((nb.shape[0], M))
    for m in range(M):
        W[:, m] = (CORRECT[m][nb] * wn).sum(1) / denom
    return W

def stageB_v5(P_rows):
    # v5 winner: DEC-space OLA-cw k=100, sharpening p=8, pure dynamic (no per-class W).
    _, idx = NN_D.kneighbors(prof(P_rows))
    Wd = comp_ola_cw(idx, 100)
    Wp = np.power(np.clip(Wd, 1e-6, None), 8)
    Wp = Wp / Wp.sum(1, keepdims=True)
    S = np.zeros(P_rows[0].shape, np.float64)
    for m in range(M):
        S += Wp[:, m:m + 1] * P_rows[m]
    return S.argmax(1)

t0 = time.time()
fam_selv = stageB_v5([OOF[m][selv_idx] for m in range(M)])
print("Stage-B family predictions for all %d SELV rows in %.0fs" % (len(selv_idx), time.time() - t0))

Stage-B family predictions for all 122469 SELV rows in 6s


## 3. Cross-fit the gate at each alpha

Same 5 folds as the pool cross-fit, so gate and pool OOF predictions are aligned row-for-row.
Probabilities are stored (not hard predictions) so thresholds can be swept without refitting.

In [4]:
ALPHAS = (0.0, 0.1, 0.25, 0.5, 0.75, 1.0)
GCACHE = Path("cache_gate_alpha.npz")

if GCACHE.exists():
    zg = np.load(GCACHE)
    GATE_OOF = {a: zg[f"a{a}"] for a in ALPHAS}
    print("loaded cached gate OOF probabilities -- delete", GCACHE, "to force a re-run")
else:
    GATE_OOF = {a: np.zeros(len(y_arr), np.float32) for a in ALPHAS}
    t0 = time.time()
    folds = list(StratifiedKFold(5, shuffle=True, random_state=RS).split(X_trf, y_arr))
    for f, (tr, va) in enumerate(folds):
        tf = time.time()
        for a in ALPHAS:
            g = fit_gate_alpha(X_trf.iloc[tr], y_arr[tr], a)
            GATE_OOF[a][va] = g.predict_proba(X_trf.iloc[va])[:, 1].astype(np.float32)
        print("  fold %d/5 done (%.0fs)" % (f + 1, time.time() - tf))
    print("gate cross-fit complete in %.0fs" % (time.time() - t0))
    np.savez_compressed(GCACHE, **{f"a{a}": GATE_OOF[a] for a in ALPHAS})

yb_arr = (y_arr != 0).astype(int)
print("\ngate quality by alpha (OOF, threshold 0.5):")
rows = []
for a in ALPHAS:
    pred = (GATE_OOF[a] >= 0.5).astype(int)
    rows.append(dict(alpha=a,
                     recall=round(recall_score(yb_arr, pred), 4),
                     precision=round(precision_score(yb_arr, pred), 4),
                     f1=round(f1_score(yb_arr, pred), 4),
                     false_pos=int(((pred == 1) & (yb_arr == 0)).sum()),
                     false_neg=int(((pred == 0) & (yb_arr == 1)).sum())))
print(pd.DataFrame(rows).to_string(index=False))

  fold 1/5 done (16s)


  fold 2/5 done (15s)


  fold 3/5 done (14s)


  fold 4/5 done (14s)


  fold 5/5 done (14s)
gate cross-fit complete in 72s



gate quality by alpha (OOF, threshold 0.5):


 alpha  recall  precision     f1  false_pos  false_neg
  0.00  0.9847     0.9294 0.9562       4857        996
  0.10  0.9864     0.9286 0.9566       4923        884
  0.25  0.9882     0.9265 0.9564       5088        764
  0.50  0.9920     0.9234 0.9565       5341        520
  0.75  0.9945     0.9209 0.9563       5543        359
  1.00  0.9961     0.9195 0.9563       5663        252


## 4. Sweep alpha x threshold on SELV, end-to-end

The metric that decides is **end-to-end 10-class macro-F1**, not gate F1: a gate improvement that
does not survive routing is not an improvement. Accuracy is reported alongside because gate precision
is the lever that moves it.

In [5]:
def route(gate_p, thr, fam_pred, ye_arr):
    yp = np.zeros(len(ye_arr), int)
    g = gate_p >= thr
    if g.any():
        yp[g] = np.vectorize(idx_to_fam.get, otypes=[int])(fam_pred[g])
    return dict(e2e_mF1=round(f1_score(ye_arr, yp, average="macro", zero_division=0), 4),
                e2e_mP=round(precision_score(ye_arr, yp, average="macro", zero_division=0), 4),
                e2e_mR=round(recall_score(ye_arr, yp, average="macro", zero_division=0), 4),
                e2e_acc=round(float((yp == ye_arr).mean()), 4))

THRS = (0.3, 0.4, 0.5, 0.6, 0.7, 0.8)
ye_selv = y_arr[selv_idx]
rows = []
for a in ALPHAS:
    gp = GATE_OOF[a][selv_idx]
    for t in THRS:
        r = route(gp, t, fam_selv, ye_selv)
        pred = (gp >= t).astype(int)
        yb_s = (ye_selv != 0).astype(int)
        r.update(alpha=a, thr=t,
                 gate_R=round(recall_score(yb_s, pred), 4),
                 gate_P=round(precision_score(yb_s, pred), 4))
        rows.append(r)
sweep = pd.DataFrame(rows)[["alpha", "thr", "gate_R", "gate_P", "e2e_mF1", "e2e_mP", "e2e_mR", "e2e_acc"]]
print(sweep.sort_values("e2e_mF1", ascending=False).head(15).to_string(index=False))

print("\n--- e2e macro-F1 grid (rows=alpha, cols=threshold) ---")
print(sweep.pivot(index="alpha", columns="thr", values="e2e_mF1").to_string())
print("\n--- e2e accuracy grid ---")
print(sweep.pivot(index="alpha", columns="thr", values="e2e_acc").to_string())

best = sweep.sort_values("e2e_mF1", ascending=False).iloc[0]
BEST_A, BEST_T = float(best["alpha"]), float(best["thr"])
base = sweep[(sweep.alpha == 1.0) & (sweep.thr == 0.5)].iloc[0]
print("\nSELV best      : alpha=%.2f thr=%.2f -> mF1 %.4f acc %.4f" % (BEST_A, BEST_T, best["e2e_mF1"], best["e2e_acc"]))
print("current default: alpha=1.00 thr=0.50 -> mF1 %.4f acc %.4f  ('balanced')" % (base["e2e_mF1"], base["e2e_acc"]))
print("gain on SELV   : %+.4f mF1, %+.4f acc   (noise band +/-0.012)"
      % (best["e2e_mF1"] - base["e2e_mF1"], best["e2e_acc"] - base["e2e_acc"]))
print("\nIs weighting redundant with thresholding? Best threshold per alpha:")
print(sweep.loc[sweep.groupby("alpha")["e2e_mF1"].idxmax()][["alpha", "thr", "e2e_mF1", "e2e_acc"]].to_string(index=False))

 alpha  thr  gate_R  gate_P  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
  0.00  0.4  0.9913  0.9246   0.6330  0.6688  0.6337   0.9101
  0.10  0.5  0.9856  0.9293   0.6328  0.6697  0.6308   0.9108
  0.75  0.6  0.9886  0.9263   0.6327  0.6692  0.6313   0.9102
  0.50  0.5  0.9918  0.9239   0.6326  0.6681  0.6332   0.9099
  0.00  0.5  0.9844  0.9300   0.6325  0.6698  0.6300   0.9108
  0.25  0.4  0.9942  0.9218   0.6324  0.6670  0.6341   0.9096
  0.50  0.6  0.9844  0.9295   0.6321  0.6700  0.6281   0.9107
  1.00  0.7  0.9848  0.9292   0.6320  0.6697  0.6280   0.9105
  0.25  0.5  0.9881  0.9273   0.6320  0.6684  0.6306   0.9104
  1.00  0.5  0.9960  0.9199   0.6317  0.6660  0.6337   0.9092
  0.10  0.6  0.9767  0.9338   0.6317  0.6710  0.6245   0.9108
  0.10  0.4  0.9921  0.9231   0.6317  0.6665  0.6332   0.9097
  0.75  0.5  0.9944  0.9215   0.6317  0.6666  0.6334   0.9095
  0.25  0.3  0.9974  0.9180   0.6316  0.6654  0.6345   0.9088
  0.50  0.4  0.9958  0.9195   0.6316  0.6657  0.6343   0.9091

--- e2e

## 5. Refit at the selected alpha and evaluate on test once

In [6]:
t0 = time.time()
Xa_f = X_trf[att_trf].reset_index(drop=True)
ya_f = pd.Series(fam_trf[att_trf])
FINAL = [fit_member(k, p, Xa_f, ya_f) for _, (k, p) in SPECS]
gate_best = fit_gate_alpha(X_trf, y_arr, BEST_A)
gate_base = fit_gate_alpha(X_trf, y_arr, 1.0)
print("pool + both gates refit in %.0fs" % (time.time() - t0))

P_test = [m.predict_proba(X_test).astype(np.float32) for m in FINAL]
fam_test = stageB_v5(P_test)
y_test_arr = y_test.values
gp_best = gate_best.predict_proba(X_test)[:, 1]
gp_base = gate_base.predict_proba(X_test)[:, 1]

res = pd.DataFrame([
    dict(config="SELV-selected: alpha=%.2f thr=%.2f" % (BEST_A, BEST_T), **route(gp_best, BEST_T, fam_test, y_test_arr)),
    dict(config="current default: balanced (alpha=1.0) thr=0.50", **route(gp_base, 0.5, fam_test, y_test_arr)),
])
print("\n--- TEST ---")
print(res.to_string(index=False))
d = res.iloc[0]["e2e_mF1"] - res.iloc[1]["e2e_mF1"]
da = res.iloc[0]["e2e_acc"] - res.iloc[1]["e2e_acc"]
print("\ngate change worth %+.4f macro-F1, %+.4f accuracy on test  (noise band +/-0.012)" % (d, da))
print("v5 reference (balanced gate): mF1 0.6363 acc 0.9089")

yb_t = (y_test_arr != 0).astype(int)
for tag, gp, t in (("selected", gp_best, BEST_T), ("balanced", gp_base, 0.5)):
    pr = (gp >= t).astype(int)
    print("  gate[%s] test recall %.4f precision %.4f | benign leaked %d"
          % (tag, recall_score(yb_t, pr), precision_score(yb_t, pr), int(((pr == 1) & (yb_t == 0)).sum())))

pool + both gates refit in 63s



--- TEST ---
                                        config  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
            SELV-selected: alpha=0.00 thr=0.40   0.6369  0.6756  0.6363   0.9100
current default: balanced (alpha=1.0) thr=0.50   0.6363  0.6733  0.6380   0.9089

gate change worth +0.0006 macro-F1, +0.0011 accuracy on test  (noise band +/-0.012)
v5 reference (balanced gate): mF1 0.6363 acc 0.9089
  gate[selected] test recall 0.9930 precision 0.9225 | benign leaked 1354
  gate[balanced] test recall 0.9966 precision 0.9175 | benign leaked 1455


### 5.1 Per-class report at the selected gate

In [7]:
yp = np.zeros(len(y_test_arr), int)
g = gp_best >= BEST_T
yp[g] = np.vectorize(idx_to_fam.get, otypes=[int])(fam_test[g])
print(classification_report(y_test_arr, yp, labels=list(range(10)),
                            target_names=NAMES, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_test_arr, yp, labels=list(range(10))),
                   index=NAMES, columns=NAMES).to_string())

v5f = dict(zip(NAMES, [0.983, 0.464, 0.623, 0.440, 0.786, 0.712, 0.817, 0.648, 0.377, 0.512]))
new = f1_score(y_test_arr, yp, average=None, labels=range(10), zero_division=0)
print("\nper-class F1 vs v5 (balanced gate):")
print(pd.DataFrame({"v5": [v5f[n] for n in NAMES], "new gate": new.round(3),
                    "delta": (new - np.array([v5f[n] for n in NAMES])).round(3)},
                   index=NAMES).to_string())

                precision    recall  f1-score   support

        Benign      0.997     0.970     0.983     45001
      Analysis      0.344     0.688     0.459        77
      Backdoor      0.762     0.533     0.627        90
           DoS      0.655     0.332     0.441       883
      Exploits      0.819     0.757     0.787      6182
       Fuzzers      0.593     0.899     0.715      5279
       Generic      0.830     0.806     0.818       899
Reconnaissance      0.749     0.573     0.649      2406
     Shellcode      0.389     0.368     0.378       370
         Worms      0.618     0.438     0.512        48

      accuracy                          0.910     61235
     macro avg      0.676     0.636     0.637     61235
  weighted avg      0.922     0.910     0.912     61235

                Benign  Analysis  Backdoor  DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
Benign           43647        43         4   11       124     1051       14              78         29 

## 6. Findings

**NULL RESULT. The gate weighting oversight was real but inconsequential. Keep the gate as it is, or
switch to alpha=0 / t=0.4 for a difference that cannot be measured.**

| Config | macro-F1 | accuracy | gate recall | gate precision | Benign leaked |
|---|---|---|---|---|---|
| SELV-selected: $\alpha=0.00$, $t=0.40$ | 0.6369 | 0.9100 | 0.9930 | 0.9225 | 1,354 |
| current default: `balanced` ($\alpha=1.0$), $t=0.50$ | 0.6363 | 0.9089 | 0.9966 | 0.9175 | 1,455 |
| **difference** | **+0.0006** | **+0.0011** | | | -101 |

Both differences are an order of magnitude inside the +/-0.012 band. Every per-class F1 delta is
<= 0.005. **Do not report 0.6369 as an improvement over v5's 0.6363** - it is the same number.

### 1. Gate weighting and gate thresholding are redundant

This is the substantive result. The best cell for *every* $\alpha$ lands within **0.0010** of every
other, and the optimal threshold shifts monotonically to compensate:

| $\alpha$ | best $t$ | e2e macro-F1 |
|---|---|---|
| 0.00 | 0.4 | 0.6330 |
| 0.10 | 0.5 | 0.6328 |
| 0.25 | 0.4 | 0.6324 |
| 0.50 | 0.5 | 0.6326 |
| 0.75 | 0.6 | 0.6327 |
| 1.00 | **0.7** | 0.6320 |

Exactly the compensating pattern this notebook was written to detect. Gate F1 is also **flat** across
$\alpha$ (0.9562-0.9566, see Section 3): re-weighting does not change the gate's ROC curve, it only
slides the operating point along it. The two knobs parameterise one degree of freedom.

**This means `two_stage_thresholds.ipynb`'s gate finding was already complete.** That notebook
concluded the gate was optimally calibrated at $p_{att} > 0.5$; the objection was that it had only
tested thresholds *under* `balanced` weighting. That objection is now answered: the conclusion holds
for every weighting in the family.

### 2. The oversight was real, and it cost nothing

`compute_sample_weight("balanced")` is exactly $\alpha=1.0$, the operating point the Stage-B weighting
study ranked **12th of 14, below no weighting at all**. That lesson was applied to Stage B and to every
pool member but never to Stage A - a genuine gap in the project's reasoning.

And $\alpha=1.0$ *is* the worst row in the grid. But the whole grid spans 0.0010, so being at the worst
point of it never mattered. **A real methodological inconsistency with no measurable consequence.**
Worth one sentence in the write-up as due diligence, not as a finding.

### 3. Why precision cannot be bought here: the leaking flows are not marginal

Dropping $\alpha$ from 1.0 to 0.0 removed **101** of 1,455 leaking Benign flows and cost gate recall
0.9966 -> 0.9930 (744 more missed attacks on the OOF set). A poor trade, and the reason is mechanical:
**the Benign flows that leak are ones the gate is confident about.** Tightening the operating point
removes the marginal cases, which were never the problem.

This reconfirms, from the weighting side, what the threshold study found from the threshold side:
*"the Fuzzers Benign-leakage finding never converted into a gain because those flows are not
threshold-separable"*. Two independent routes to the same conclusion.

The Benign row of the confusion matrix makes it concrete: **1,051 of the 1,354 remaining leaks still go
to Fuzzers.** That is a *representation* problem - those flows look like Fuzzers in CICFlowMeter feature
space - and no gate operating point can fix it.

### 4. Accuracy moved in the predicted direction, and it was not enough

Accuracy was the one metric this lever was expected to move, since it is dominated by the 73.5% Benign
class and gate false positives are what damage it. It did move: 0.9089 -> 0.9100 (**+0.0011**), the
largest accuracy gain of any configuration tried. It is still noise. The accuracy grid peaks at
0.9108 ($\alpha \in \{0, 0.1\}$, $t=0.5$) against 0.9079 at the worst corner - a total span of 0.0029
across the entire sweep.

### Recommendation

**Close option C.** The gate is at its achievable optimum and has been throughout the project. If a
change is made at all, prefer $\alpha=0$ with $t=0.5$ (accuracy 0.9108 on SELV, simplest and best
accuracy) over `balanced` - but state plainly that the difference is unmeasurable.

**Gate precision is closed as a lever, by two independent routes.** Combined with the v5 finding that
competence is 98.1% row difficulty, the remaining error is now bounded on both sides of the hierarchy:

- **Stage A**: on a fixed ROC curve, already at its optimum; residual leakage is not separable.
- **Stage B**: competence is estimable but measures row difficulty, not member choice.

Both point at the same place. **Only option D (derived features) remains** - the one lever that moves
the Cover-Hart bound rather than approaching it - together with **SHAP**, which is a contribution gap
rather than a score lever.